# ML Lab:

The objective is to build an MLP to perform classification of industrial motors between 7 classes (0: healthy, 1-6: levels of degradation). Specifically, the issue treated here is the problem of inter-turn short circuits on 3-phase motors.

## 1. EDA:
The [dataset](https://www.kaggle.com/datasets/rebecacunha/mit-short-circuit-flux-and-current-signals) used contains 2618 csv files, each containing the recordings of the variations of the three phases courrents (CH1, CH2, CH3) plus the magnetic flux (CH4) per time.

Below is data for an example of a healthy motor


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
healthy_data_sample = pd.read_csv('../data/raw/0_c2264c30c100.csv', skiprows=20)
healthy_data_sample.info()
healthy_data_sample.head()

In [ ]:
faulty_data_sample = pd.read_csv('../data/raw/5_c1437c60c100.csv', skiprows = 20)

In [ ]:
fig, axs = plt.subplots(2, 2)

axs[0, 0].plot(healthy_data_sample['TIME'][:5000], healthy_data_sample['CH1'][:5000], color='red')
axs[0, 1].plot(healthy_data_sample['TIME'][:5000], healthy_data_sample['CH4'][:5000], color='blue')
axs[1, 0].plot(faulty_data_sample['TIME'][:5000], faulty_data_sample['CH1'][:5000], color='green')
axs[1, 1].plot(faulty_data_sample['TIME'][:5000], faulty_data_sample['CH4'][:5000], color='purple')

axs[0, 0].set_title('Healthy Motor Phase A Current')
axs[0, 1].set_title('Healthy Motor Magnetic Flux')
axs[1, 0].set_title('Faulty Motor Phase A Current')
axs[1, 1].set_title('Faulty Motor Magnetic Flux')

fig.suptitle('Healthy vs. Faulty (lvl5) Motors')
fig.tight_layout()
plt.show()

## 2. Feature Engineering:
as MLPs cannot fit well the data in its current shape (continous on time, while the MLPs treat each point individually without the proximity/neigbhor context), we propose extracting the following features from the csv files:

| Feature Category | Feature Name                                                                                       | Applied To              | Total Column Count |
| ---------------- | -------------------------------------------------------------------------------------------------- | ----------------------- | ------------------ |
| Time Domain      | "Mean, Std, RMS, Peak2Peak, Skewness, Kurtosis, Crest, Form, Clearance, ZCR, Energy"               | "ia​,ib​,ic​, and Flux" | 11×4=44            |
| Frequency Domain | "Harmonic Magnitudes (3rd, 5th, 7th), THD, Spectral Centroid, Spectral Spread"                     | "ia​,ib​,ic​, and Flux" | 6×4=24             |
| Frequency Domain | Sideband Ratio                                                                                     | "ia​,ib​,ic​ only"      | 1×3=3              |
| Global System    | "Fundamental Freq, Phase Imbalance, Park Vector Stats (x3), Flux-Current Phase, Negative Sequence" | Combined System         | 7                  |
|  Total           |                                                                                                    |                         | 78 Features        |

We will run the extraction feature script and create a new csv file called features.csv which will contain all the new features for all 2618 files.

The result is as follows:

In [ ]:
feat = pd.read_csv('../data/processed/features.csv')
feat.info()
feat.head(n=10)

## 3. Data Cleaning and Preparation:

### 3.1 Data Splitting and Handling Missing Values:

We found some NULL/NA values on our new dataset due to correpted files, errors...etc

In [ ]:
feat.isna().sum()

First, We will follow the 70/30 train-test split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = feat.drop('class', axis=1)
y = feat['class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test = X_test.fillna(X_test.median(numeric_only=True))

In [ ]:
X_train.isna().sum()

In [ ]:
X_test.isna().sum()


### 3.2 Data Scaling:


Then, we perform scaling 

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. MLP training:

Now we are ready to train the MLP on the data

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:

mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
mlp.fit(X_train_scaled, y_train)

## 5. Model Evaluation:

In [ ]:
y_pred = mlp.predict(X_test_scaled)
print("Test Accuracy:", accuracy_score(y_test, y_pred))

y_pred2 = mlp.predict(X_train_scaled)
print("Train Accuracy:", accuracy_score(y_train, y_pred2))

print(classification_report(y_test, y_pred))

## 6. Conclusion:

In this lab, we performed the ITSC detection using an MLP model. The model passed the accuracy test with almost 97% on both train and test sets, which indicates a succeessful model.